# LAB | Hyperparameter Tuning

**Load the data**

Finally step in order to maximize the performance on your Spaceship Titanic model.

The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

So far we've been training and evaluating models with default values for hyperparameters.

Today we will perform the same feature engineering as before, and then compare the best working models you got so far, but now fine tuning it's hyperparameters.

In [1]:
#Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
# Revisar tamaño del dataset
spaceship.shape


(8693, 14)

In [4]:
# Revisar tipos de datos
spaceship.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


In [5]:
# Revisar valores nulos
spaceship.isna().sum()


PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [6]:
# Eliminar filas con nulos
spaceship_clean = spaceship.dropna().copy()

spaceship_clean.isna().sum()


PassengerId     0
HomePlanet      0
CryoSleep       0
Cabin           0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Name            0
Transported     0
dtype: int64

In [7]:
# Transformar Cabin
spaceship_clean["Cabin"] = spaceship_clean["Cabin"].str.split("/").str[0]

spaceship_clean["Cabin"].value_counts()


Cabin
F    2152
G    1973
E     683
B     628
C     587
D     374
A     207
T       2
Name: count, dtype: int64

In [8]:
# Eliminar PassengerId y Name
spaceship_clean = spaceship_clean.drop(columns=["PassengerId", "Name"])

spaceship_clean.head()


,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported
0,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,False
1,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,True
2,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,False
3,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,False
4,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,True


In [9]:
# Convertir columnas categóricas en dummies
spaceship_encoded = pd.get_dummies(spaceship_clean, drop_first=True)

spaceship_encoded.head()


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,39.0,0.0,0.0,0.0,0.0,0.0,False,True,False,False,True,False,False,False,False,False,False,False,True,False
1,24.0,109.0,9.0,25.0,549.0,44.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False
2,58.0,43.0,3576.0,0.0,6715.0,49.0,False,True,False,False,False,False,False,False,False,False,False,False,True,True
3,33.0,0.0,1283.0,371.0,3329.0,193.0,False,True,False,False,False,False,False,False,False,False,False,False,True,False
4,16.0,303.0,70.0,151.0,565.0,2.0,True,False,False,False,False,False,False,False,True,False,False,False,True,False


In [10]:
# Separar features y target
X = spaceship_encoded.drop(columns=["Transported"])
y = spaceship_encoded["Transported"]

print("X shape:", X.shape)
print("y shape:", y.shape)



X shape: (6606, 19)
y shape: (6606,)


Now perform the same as before:
- Feature Scaling
- Feature Selection


In [11]:
# Escalar variables
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
   X_scaled,
   columns=X.columns,
   index=X.index
)

X_scaled.head()


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.695413,-0.345756,-0.285355,-0.309494,-0.273759,-0.269534,1.717147,-0.510811,-0.738664,3.085305,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
1,-0.336769,-0.176748,-0.279993,-0.266112,0.206165,-0.230494,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
2,2.002842,-0.279083,1.845163,-0.309494,5.596357,-0.226058,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,6.306963
3,0.282540,-0.345756,0.479034,0.334285,2.636384,-0.098291,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
4,-0.887266,0.124056,-0.243650,-0.047470,0.220152,-0.267759,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555


In [12]:
# Seleccionar mejores variables
selector = SelectKBest(score_func=f_classif, k=20)

X_selected = selector.fit_transform(X_scaled, y)

selected_features = X_scaled.columns[selector.get_support()]

X_selected = pd.DataFrame(
   X_selected,
   columns=selected_features,
   index=X_scaled.index
)

X_selected.head()



/Users/gabrielbohorquez/Library/Python/3.9/lib/python/site-packages/sklearn/feature_selection/_univariate_selection.py:783: UserWarning: k=20 is greater than n_features=19. All the features will be returned.
  warnings.warn(


,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
0,0.695413,-0.345756,-0.285355,-0.309494,-0.273759,-0.269534,1.717147,-0.510811,-0.738664,3.085305,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
1,-0.336769,-0.176748,-0.279993,-0.266112,0.206165,-0.230494,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
2,2.002842,-0.279083,1.845163,-0.309494,5.596357,-0.226058,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,6.306963
3,0.282540,-0.345756,0.479034,0.334285,2.636384,-0.098291,1.717147,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,-0.695098,-0.652578,-0.017402,-0.322689,0.666047,-0.158555
4,-0.887266,0.124056,-0.243650,-0.047470,0.220152,-0.267759,-0.582361,-0.510811,-0.738664,-0.324117,-0.312289,-0.244975,-0.339578,1.438646,-0.652578,-0.017402,-0.322689,0.666047,-0.158555


In [13]:
# Ver variables seleccionadas
selected_features


Index(['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck',
       'HomePlanet_Europa', 'HomePlanet_Mars', 'CryoSleep_True', 'Cabin_B',
       'Cabin_C', 'Cabin_D', 'Cabin_E', 'Cabin_F', 'Cabin_G', 'Cabin_T',
       'Destination_PSO J318.5-22', 'Destination_TRAPPIST-1e', 'VIP_True'],
      dtype='object')

In [14]:
# Dividir en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(
   X_selected,
   y,
   test_size=0.2,
   random_state=42,
   stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (5284, 19)
X_test: (1322, 19)
y_train: (5284,)
y_test: (1322,)


In [15]:
# Entrenar modelo base
base_model = RandomForestClassifier(
   random_state=42
)

base_model.fit(X_train, y_train)


RandomForestClassifier(random_state=42)

- Now let's use the best model we got so far in order to see how it can improve when we fine tune it's hyperparameters.

In [ ]:
#your code here

- Evaluate your model

In [16]:
# Evaluar modelo base
base_predictions = base_model.predict(X_test)

base_accuracy = accuracy_score(y_test, base_predictions)

print(f"Accuracy modelo base: {base_accuracy:.4f}")
print("\nMatriz de confusión:")
display(confusion_matrix(y_test, base_predictions))

print("\nClassification report:")
print(classification_report(y_test, base_predictions))


Accuracy modelo base: 0.7806

Matriz de confusión:


array([[512, 144],
       [146, 520]])


Classification report:
              precision    recall  f1-score   support

       False       0.78      0.78      0.78       656
        True       0.78      0.78      0.78       666

    accuracy                           0.78      1322
   macro avg       0.78      0.78      0.78      1322
weighted avg       0.78      0.78      0.78      1322



**Grid/Random Search**

For this lab we will use Grid Search.

- Define hyperparameters to fine tune.

In [17]:
# Definir grid de hiperparámetros
param_grid = {
   "n_estimators": [100, 200],
   "max_depth": [None, 10, 20],
   "min_samples_split": [2, 5],
   "min_samples_leaf": [1, 2],
   "max_features": ["sqrt", "log2"]
}


- Run Grid Search

In [18]:
# Ejecutar Grid Search
grid_search = GridSearchCV(
   estimator=RandomForestClassifier(random_state=42),
   param_grid=param_grid,
   scoring="accuracy",
   cv=5,
   n_jobs=-1,
   verbose=1
)

grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 48 candidates, totalling 240 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 10, 20],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200]},
             scoring='accuracy', verbose=1)

In [19]:
# Ver mejores hiperparámetros
print("Mejores hiperparámetros:")
print(grid_search.best_params_)

print("\nMejor accuracy en validación cruzada:")
print(grid_search.best_score_)


Mejores hiperparámetros:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}

Mejor accuracy en validación cruzada:
0.8033707462515409


- Evaluate your model

In [20]:
# Evaluar mejor modelo
best_model = grid_search.best_estimator_

tuned_predictions = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test, tuned_predictions)

print(f"Accuracy modelo base: {base_accuracy:.4f}")
print(f"Accuracy modelo optimizado: {tuned_accuracy:.4f}")

print("\nMatriz de confusión modelo optimizado:")
display(confusion_matrix(y_test, tuned_predictions))

print("\nClassification report modelo optimizado:")
print(classification_report(y_test, tuned_predictions))


Accuracy modelo base: 0.7806
Accuracy modelo optimizado: 0.7927

Matriz de confusión modelo optimizado:


array([[503, 153],
       [121, 545]])


Classification report modelo optimizado:
              precision    recall  f1-score   support

       False       0.81      0.77      0.79       656
        True       0.78      0.82      0.80       666

    accuracy                           0.79      1322
   macro avg       0.79      0.79      0.79      1322
weighted avg       0.79      0.79      0.79      1322



In [21]:
# Comparación final
comparison_results = pd.DataFrame({
   "Model": ["Random Forest Base", "Random Forest Tuned"],
   "Accuracy": [base_accuracy, tuned_accuracy]
})

comparison_results


,Model,Accuracy
0,Random Forest Base,0.780635
1,Random Forest Tuned,0.792738


## Conclusión final

En este laboratorio se aplicó hyperparameter tuning usando Grid Search para mejorar el rendimiento del modelo Random Forest.

Primero se preparó el dataset Spaceship Titanic aplicando el mismo flujo anterior: limpieza de nulos, transformación de `Cabin`, eliminación de columnas identificadoras, codificación de variables categóricas, escalado y selección de variables.

Después se entrenó un modelo base de Random Forest con hiperparámetros por defecto y se evaluó su rendimiento sobre el conjunto de test.

Luego se definió una búsqueda de hiperparámetros usando Grid Search. Esta técnica probó diferentes combinaciones de parámetros como `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf` y `max_features`.

Finalmente se comparó el modelo base con el modelo optimizado. Si el modelo optimizado obtiene mayor accuracy, significa que el ajuste de hiperparámetros mejoró el rendimiento. Si la mejora es pequeña o inexistente, puede indicar que el modelo base ya funcionaba razonablemente bien o que sería necesario ampliar el grid de búsqueda.

En un contexto profesional, no elegiríamos el modelo solo por accuracy. También revisaríamos precision, recall, F1-score y el objetivo de negocio.
